In [11]:
!pip install torch transformers google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client pyvi sentencepiece

In [15]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from pyvi import ViTokenizer

# ==========================================
# ☁️ 1. KÉO NÃO AI TỪ HUGGING FACE VỀ
# ==========================================
print("Đang tải AI từ đám mây về... Lần đầu sẽ hơi lâu một chút để tải file!")

# Tên repo chứa model đã train
MODEL_PATH = "hoangkkk/phobert-v2-vietnamese-sentiment"

# 1. Tải Tokenizer chuẩn từ kho gốc của VinAI (Đảm bảo không bao giờ thiếu file)
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2", use_fast=False)

# 2. Tải Model (Trọng số đã train) từ kho của hoangkkk
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

def segment_text(text):
    return ViTokenizer.tokenize(str(text))

def predict_toxic(text):
    segmented_text = segment_text(text)
    inputs = tokenizer(segmented_text, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
    prediction = torch.argmax(outputs.logits, dim=-1).item()
    return prediction == 1

print("✅ Đã nạp não xong! Mô hình sẵn sàng chiến đấu.\n")
# ==========================================
# 🤖 2. KẾT NỐI API YOUTUBE (Như cũ)
# ==========================================
SCOPES = ["https://www.googleapis.com/auth/youtube.force-ssl"]

def get_authenticated_service():
    os.environ['OAUTHLIB_INSECURE_TRANSPORT'] = '1'
    flow = InstalledAppFlow.from_client_secrets_file("client_secret.json", SCOPES, redirect_uri='http://localhost:8080/')
    auth_url, _ = flow.authorization_url(prompt='consent')

    print("👉 Bấm vào link để đăng nhập Google: ", auth_url)
    redirect_response = input('👉 DÁN ĐƯỜNG LINK LỖI VÀO ĐÂY VÀ ENTER: ')
    flow.fetch_token(authorization_response=redirect_response.strip())
    return build("youtube", "v3", credentials=flow.credentials)

def moderate_youtube_comments(youtube, video_id):
    request = youtube.commentThreads().list(
        part="snippet",
        videoId=video_id,
        textFormat="plainText"
    )
    response = request.execute()

    for item in response.get("items", []):
        comment_id = item["id"]
        comment_text = item["snippet"]["topLevelComment"]["snippet"]["textDisplay"]
        author = item["snippet"]["topLevelComment"]["snippet"]["authorDisplayName"]

        print(f"[{author}]: {comment_text}")

        # ==========================================
        # ⚔️ 3. AI CHÉM BÌNH LUẬN
        # ==========================================
        if predict_toxic(comment_text):
            print("   => ⚠️ AI phán: TOXIC! Đang dọn dẹp...")
            try:
                youtube.comments().setModerationStatus(
                    id=comment_id,
                    moderationStatus="heldForReview" # Ẩn đi chờ bạn duyệt lại
                ).execute()
                print("   => 🟢 Đã ẩn thành công!\n")
            except Exception as e:
                print(f"   => 🔴 Lỗi API YouTube: {e}\n")
        else:
            print("   => 🟢 AI phán: Bình luận sạch.\n")

if __name__ == "__main__":
    youtube_service = get_authenticated_service()

    # 👉 THAY ID VIDEO CỦA BẠN VÀO ĐÂY
    VIDEO_ID = "cxdU2ciJUUQ"

    moderate_youtube_comments(youtube_service, VIDEO_ID)


Đang tải AI từ đám mây về... Lần đầu sẽ hơi lâu một chút để tải file!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Đã nạp não xong! Mô hình sẵn sàng chiến đấu.

👉 Bấm vào link để đăng nhập Google:  https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=767804715875-1cb1ge8r6c8oih4ibp7phpqrdhul5kq3.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fyoutube.force-ssl&state=a8NSUubzu1ZqZsAhOiXe1M8B4GtSOb&code_challenge=gQzpZ0Sbj3nwOrU_o8zVEdSYKyklmlc9eydFOSzWkpY&code_challenge_method=S256&prompt=consent&access_type=offline
👉 DÁN ĐƯỜNG LINK LỖI VÀO ĐÂY VÀ ENTER: http://localhost:8080/?state=a8NSUubzu1ZqZsAhOiXe1M8B4GtSOb&iss=https://accounts.google.com&code=4/0Aci98E8j_zbZUzATCSPjVPM-ihVWCBnS7U7NYpbzPTFHhg1cdKhCsLaKy4jHDxTGjrO7CQ&scope=https://www.googleapis.com/auth/youtube.force-ssl
[@hoangvu4327]: súc vật
   => ⚠️ AI phán: TOXIC! Đang dọn dẹp...
   => 🟢 Đã ẩn thành công!

[@hoangvu4327]: khôn thế
   => 🟢 AI phán: Bình luận sạch.

[@hoangvu4327]: lọc bình luận
   => 🟢 AI phán: Bình luận sạch.

[@hoangvu4327]: test